# Arithmetic Coding Example

This notebook walks through the arithmetic coding process for the message `"ARITHMETIC"`.
We compute symbol probabilities from the message itself, build the cumulative interval
assigned to each symbol, and then update the encoding interval one character at a time.


In [1]:
from collections import Counter

message = "ARITHMETIC"
alphabet = sorted(set(message))
frequencies = Counter(message)
message_length = len(message)

print(f"Message: {message}")
print(f"Total symbols: {message_length}")
print("\nSymbol model based on message frequencies:")
header = f"{'Symbol':<8}{'Count':>8}{'Probability':>15}{'Cumulative Low':>20}{'Cumulative High':>20}"
print(header)
print('-' * len(header))

cumulative = 0.0
symbol_intervals = {}
for symbol in alphabet:
    probability = frequencies[symbol] / message_length
    low = cumulative
    high = cumulative + probability
    symbol_intervals[symbol] = (low, high)
    print(f"{symbol:<8}{frequencies[symbol]:>8}{probability:>15.6f}{low:>20.6f}{high:>20.6f}")
    cumulative = high

if abs(cumulative - 1.0) > 1e-9:
    print("Warning: cumulative probabilities do not sum to 1.0")


Message: ARITHMETIC
Total symbols: 10

Symbol model based on message frequencies:
Symbol     Count    Probability      Cumulative Low     Cumulative High
-----------------------------------------------------------------------
A              1       0.100000            0.000000            0.100000
C              1       0.100000            0.100000            0.200000
E              1       0.100000            0.200000            0.300000
H              1       0.100000            0.300000            0.400000
I              2       0.200000            0.400000            0.600000
M              1       0.100000            0.600000            0.700000
R              1       0.100000            0.700000            0.800000
T              2       0.200000            0.800000            1.000000


In [2]:
print("\nStep-by-step interval updates:")
header = (
    f"{'Step':>4}{'Symbol':>10}{'Prev Low':>20}{'Prev High':>20}{'Range':>20}"
    f"{'Symbol Low':>15}{'Symbol High':>15}{'New Low':>20}{'New High':>20}"
)
print(header)
print('-' * len(header))

low, high = 0.0, 1.0
for index, symbol in enumerate(message, 1):
    range_width = high - low
    symbol_low, symbol_high = symbol_intervals[symbol]
    new_low = low + range_width * symbol_low
    new_high = low + range_width * symbol_high
    print(
        f"{index:>4}{symbol:>10}{low:>20.12f}{high:>20.12f}{range_width:>20.12f}"
        f"{symbol_low:>15.6f}{symbol_high:>15.6f}{new_low:>20.12f}{new_high:>20.12f}"
    )
    low, high = new_low, new_high

code_value = (low + high) / 2
print("\nFinal interval:")
print(f"  low  = {low:.12f}")
print(f"  high = {high:.12f}")
print(f"A valid code value inside the final interval is {code_value:.12f}.")



Step-by-step interval updates:
Step    Symbol            Prev Low           Prev High               Range     Symbol Low    Symbol High             New Low            New High
------------------------------------------------------------------------------------------------------------------------------------------------
   1         A      0.000000000000      1.000000000000      1.000000000000       0.000000       0.100000      0.000000000000      0.100000000000
   2         R      0.000000000000      0.100000000000      0.100000000000       0.700000       0.800000      0.070000000000      0.080000000000
   3         I      0.070000000000      0.080000000000      0.010000000000       0.400000       0.600000      0.074000000000      0.076000000000
   4         T      0.074000000000      0.076000000000      0.002000000000       0.800000       1.000000      0.075600000000      0.076000000000
   5         H      0.075600000000      0.076000000000      0.000400000000       0.300000       0.